# Reservoir Inflow Forecasting — 3-Model Comparison (CatBoost / XGBoost / LightGBM)
## Split แบบใหม่ (Stratified / Season-Balanced)

เปรียบเทียบ 3 โมเดล ML บน **split แบบใหม่ที่ stratify ตามฤดู** (ปรับให้ zero-inflow rate ใกล้เคียงกันใน train/val/test มากขึ้น) เทียบกับ notebook อีกไฟล์ที่ใช้ split แบบเดิม เพื่อดูว่า:
1. การแก้ split ช่วยให้ Val_NSE สมเหตุสมผลขึ้นจริงหรือไม่ (ตามที่พบในรอบทดสอบ CatBoost เดี่ยวก่อนหน้า)
2. ผลสรุปว่าโมเดลไหนดีที่สุดเปลี่ยนไปหรือไม่เมื่อ split เปลี่ยน

**โมเดลที่เทียบ:** CatBoost, XGBoost, และ **LightGBM** (เลือกเป็นตัวที่ 3 เพราะเป็นหนึ่งใน gradient-boosted tree มาตรฐาน 3 ตัวที่ใช้เทียบกันทั่วไปในงาน hydrology ML, ใช้วิธี split-finding ที่ต่างจากอีก 2 ตัว (leaf-wise histogram-based), และรองรับ SHAP TreeExplainer + Optuna ได้สมบูรณ์)

**องค์ประกอบที่ใช้ครบทุกอย่างเหมือน pipeline เดิม:**
- Leakage-safe split + early stopping (val เท่านั้น, test แตะครั้งเดียว)
- Walk-forward CV วินิจฉัยความเสถียร (train block เท่านั้น)
- Bayesian hyperparameter search (Optuna/TPE) **แยกต่อโมเดลต่อ horizon**
- SHAP feature importance
- KGE metric เสริมจาก NSE/RMSE/MAE

> **ข้อจำกัดที่ควรรู้ก่อนอ่านผล:** Stratified split นี้เป็นการประนีประนอม ไม่ใช่การแก้ปัญหาที่สมบูรณ์ — train ยังมี zero-rate ต่ำกว่า val/test อยู่บ้าง (เพราะ train ต้องรับวันฤดูฝนที่ไม่มี zero เลยจำนวนมาก) และ test set ใหม่มีองค์ประกอบ/scale ต่างจาก test set เดิม ทำให้ **เทียบ RMSE ข้าม notebook ตรงๆไม่ได้** ต้องเทียบ Model vs Baseline ภายใน notebook เดียวกันเท่านั้น รายละเอียดเต็มอยู่ใน Part 1


## Setup

In [1]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import optuna
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

OUTDIR = "outputs_stratified"
os.makedirs(OUTDIR, exist_ok=True)

## Model Adapters

อินเทอร์เฟซเดียวกันสำหรับ CatBoost / XGBoost / LightGBM เพื่อให้โค้ดส่วน CV, Optuna, SHAP, และ scoring ใช้ร่วมกันได้โดยไม่ต้องเขียนแยกทีละโมเดล


In [2]:
# =============================================================================
# Model adapter: one common interface over CatBoost / XGBoost / LightGBM
# =============================================================================
# Each adapter exposes: build(params) -> fresh estimator
#                        fit(model, X_tr, y_tr, X_val, y_val) -> fitted model, best_iter
#                        predict(model, X) -> np.ndarray
# so the rest of the pipeline (CV diagnostic, Optuna objective, final test
# scoring, SHAP) does not need to know which library it is calling.

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import lightgbm as lgb


class CatBoostAdapter:
    name = "CatBoost"

    @staticmethod
    def build(params, with_early_stopping=True):
        p = dict(params)
        p.setdefault("loss_function", "MAE")
        p.setdefault("bootstrap_type", "Bernoulli")
        p.setdefault("random_seed", 42)
        p.setdefault("verbose", False)
        if with_early_stopping:
            p.setdefault("od_type", "Iter")
            p.setdefault("od_wait", 200)
        return CatBoostRegressor(**p)

    @staticmethod
    def fit(model, X_tr, y_tr, X_val=None, y_val=None):
        if X_val is not None:
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True, verbose=False)
            best_iter = model.get_best_iteration()
        else:
            model.fit(X_tr, y_tr, verbose=False)
            best_iter = model.tree_count_
        return model, best_iter

    @staticmethod
    def predict(model, X):
        return model.predict(X)


class XGBoostAdapter:
    name = "XGBoost"

    @staticmethod
    def build(params, with_early_stopping=True):
        p = dict(params)
        p.setdefault("random_state", 42)
        p.setdefault("verbosity", 0)
        if with_early_stopping:
            p.setdefault("early_stopping_rounds", 50)
            p.setdefault("eval_metric", "rmse")
        return XGBRegressor(**p)

    @staticmethod
    def fit(model, X_tr, y_tr, X_val=None, y_val=None):
        if X_val is not None:
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            best_iter = model.best_iteration if hasattr(model, "best_iteration") and model.best_iteration is not None else model.n_estimators
        else:
            model.fit(X_tr, y_tr, verbose=False)
            best_iter = model.n_estimators
        return model, best_iter

    @staticmethod
    def predict(model, X):
        return model.predict(X)


class LightGBMAdapter:
    name = "LightGBM"

    @staticmethod
    def build(params, with_early_stopping=True):
        p = dict(params)
        p.setdefault("random_state", 42)
        p.setdefault("verbose", -1)
        return LGBMRegressor(**p)

    @staticmethod
    def fit(model, X_tr, y_tr, X_val=None, y_val=None):
        if X_val is not None:
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric="rmse",
                       callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
            best_iter = model.best_iteration_
        else:
            model.fit(X_tr, y_tr)
            best_iter = model.n_estimators_
        return model, best_iter

    @staticmethod
    def predict(model, X):
        return model.predict(X)


MODEL_ADAPTERS = {
    "CatBoost": CatBoostAdapter,
    "XGBoost": XGBoostAdapter,
    "LightGBM": LightGBMAdapter,
}

# Default (un-tuned) parameter sets per model, used for the "baseline
# hyperparameters" pass before Optuna search, kept in the same spirit as
# the original cat_params from the single-model pipeline.
DEFAULT_PARAMS = {
    "CatBoost": dict(
        iterations=1500, learning_rate=0.06, depth=3, l2_leaf_reg=350,
        min_data_in_leaf=30, subsample=0.5, rsm=0.9, random_strength=1.0,
    ),
    "XGBoost": dict(
        n_estimators=1500, learning_rate=0.06, max_depth=3, reg_lambda=350,
        min_child_weight=30, subsample=0.5, colsample_bytree=0.9,
    ),
    "LightGBM": dict(
        n_estimators=1500, learning_rate=0.06, max_depth=3, reg_lambda=350,
        min_child_samples=30, subsample=0.5, colsample_bytree=0.9,
        subsample_freq=1,
    ),
}

# Optuna search spaces per model -- analogous ranges, translated to each
# library's native parameter names.
def make_optuna_params(trial, model_name):
    if model_name == "CatBoost":
        return dict(
            iterations=trial.suggest_int("iterations", 300, 1500, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            depth=trial.suggest_int("depth", 2, 6),
            l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 500.0, log=True),
            min_data_in_leaf=trial.suggest_int("min_data_in_leaf", 5, 40),
            subsample=trial.suggest_float("subsample", 0.4, 1.0),
            rsm=trial.suggest_float("rsm", 0.5, 1.0),
            random_strength=trial.suggest_float("random_strength", 0.0, 3.0),
        )
    elif model_name == "XGBoost":
        return dict(
            n_estimators=trial.suggest_int("n_estimators", 300, 1500, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            max_depth=trial.suggest_int("max_depth", 2, 6),
            reg_lambda=trial.suggest_float("reg_lambda", 1.0, 500.0, log=True),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 40),
            subsample=trial.suggest_float("subsample", 0.4, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        )
    elif model_name == "LightGBM":
        return dict(
            n_estimators=trial.suggest_int("n_estimators", 300, 1500, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            max_depth=trial.suggest_int("max_depth", 2, 6),
            reg_lambda=trial.suggest_float("reg_lambda", 1.0, 500.0, log=True),
            min_child_samples=trial.suggest_int("min_child_samples", 5, 40),
            subsample=trial.suggest_float("subsample", 0.4, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            subsample_freq=1,  # required for LightGBM to actually use `subsample`
        )
    else:
        raise ValueError(model_name)

## Part 1 — Load, audit, **stratified (season-balanced) split**

**ปัญหาที่พบมาก่อน:** ข้อมูล 1 ปีนี้มี zero-inflow (Q_in_t = 0) ทั้งหมด 45/359 วัน กระจุกอยู่ในช่วงเดียว (2026-02-03 ถึง 2026-05-17, 104 วัน) ไม่มีวัน zero เลยนอกช่วงนี้ การแบ่ง chronological ตรงไปตรงมาทำให้ val ได้ฤดูแล้งไปเกือบทั้งหมด (zero-rate ~58%) ในขณะที่ train มีแค่ ~3% — ทำให้ early stopping และ Val_NSE ผิดเพี้ยนหนักไม่ว่าจะใช้ hyperparameter ไหน

**วิธีแก้ที่ใช้:** แบ่งปีออกเป็น 3 ช่วง (wet_before, dry_window, wet_after) แล้วจัดสรร dry_window แบบ **ตามสัดส่วน 70/15/15** (ไม่ใช่ equal thirds — equal thirds ทำให้ train เจือจางเกินไปเพราะต้องรับ wet_before 221 วันที่ไม่มี zero เลย) ผลคือ zero-rate ของแต่ละ partition ใกล้เคียงกันขึ้นมาก (~12% / ~18-20% / ~8-9%) แต่ **ยังไม่เท่ากันสนิท** เพราะ dry days ในข้อมูลไม่ได้กระจายสม่ำเสมอ — ดูตัวเลขจริงด้านล่างก่อนตีความผลใดๆ


In [3]:
df = pd.read_csv("Training_Values_Nofct_7day_CORRECTED.csv")
df["Date_dt"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")
df = df.sort_values("Date_dt").reset_index(drop=True)

print("Date range:", df["Date_dt"].min(), "to", df["Date_dt"].max())
print("Rows:", len(df), "| Cols:", df.shape[1])

DATE_COL = "Date_dt"
QIN_COL = "Q_in_t (m3/day)"
RAIN_COL = "Rain_obs_t (mm)"
API_COL = "API_t (mm)"
for c in [QIN_COL, RAIN_COL, API_COL]:
    assert c in df.columns, f"Expected column not found: {c}"

targets = [c for c in df.columns if re.match(r"^y\d+=Q_in", str(c))]
targets = sorted(targets, key=lambda x: int(re.findall(r"^y(\d+)=", x)[0]))
print("Targets:", targets)

exclude = {"Date", "Date_num", "Date_dt"} | set(targets)
feature_cols = [c for c in df.columns if c not in exclude]
print("Features:", feature_cols)

n_nan_deltaS = df["DeltaS_t (m3/day)"].isna().sum()
if n_nan_deltaS > 0:
    print(f"Forward-filling {n_nan_deltaS} NaN value(s) in DeltaS_t (documented limitation).")
    df["DeltaS_t (m3/day)"] = df["DeltaS_t (m3/day)"].ffill()

X = df[feature_cols].copy()
Y = df[targets].copy()
mask = (~Y.isna().any(axis=1)) & (~df[DATE_COL].isna())
X, Y = X.loc[mask].reset_index(drop=True), Y.loc[mask].reset_index(drop=True)
dates = df.loc[mask, DATE_COL].reset_index(drop=True)
N = len(dates)
print("Rows used:", len(X), "| Date range:", dates.min(), "to", dates.max())

Date range: 2025-06-27 00:00:00 to 2026-06-20 00:00:00
Rows: 359 | Cols: 16
Targets: ['y1=Q_in_t+1 (m3/day)', 'y2=Q_in_t+2 (m3/day)', 'y3=Q_in_t+3 (m3/day)', 'y4=Q_in_t+4 (m3/day)', 'y5=Q_in_t+5 (m3/day)', 'y6=Q_in_t+6 (m3/day)', 'y7=Q_in_t+7 (m3/day)']
Features: ['Q_in_t (m3/day)', 'Water_Level_t (m)', 'Storage_S_t (m3)', 'DeltaS_t (m3/day)', '%Full_t', 'Rain_obs_t (mm)', 'API_t (mm)']
Forward-filling 1 NaN value(s) in DeltaS_t (documented limitation).
Rows used: 359 | Date range: 2025-06-27 00:00:00 to 2026-06-20 00:00:00


In [4]:
ZERO_THRESHOLD = 1e-6

dry_start, dry_end = pd.Timestamp("2026-02-03"), pd.Timestamp("2026-05-17")
in_dry = (dates >= dry_start) & (dates <= dry_end)
wet_before_idx = np.where(dates < dry_start)[0]
dry_idx = np.where(in_dry)[0]
wet_after_idx = np.where(dates > dry_end)[0]

print(f"Segments: wet_before={len(wet_before_idx)} days | dry_window={len(dry_idx)} days "
      f"({(Y[targets[0]].to_numpy()[dry_idx] <= ZERO_THRESHOLD).sum()} zero) | wet_after={len(wet_after_idx)} days")

# Dry window allocated PROPORTIONALLY to 70/15/15 (not equal thirds -- see
# Part 1 markdown for why equal thirds under-serves train's zero-rate).
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15
n_dry = len(dry_idx)
n_dry_train = round(n_dry * TRAIN_FRAC)
n_dry_val = round(n_dry * VAL_FRAC)
dry_train_idx = dry_idx[:n_dry_train]
dry_val_idx = dry_idx[n_dry_train:n_dry_train + n_dry_val]
dry_test_idx = dry_idx[n_dry_train + n_dry_val:]

print(f"Dry window allocated proportionally: train={len(dry_train_idx)}, val={len(dry_val_idx)}, test={len(dry_test_idx)}")

def chrono_split(idx_array, frac_train=TRAIN_FRAC, frac_val=VAL_FRAC):
    n = len(idx_array)
    c1 = int(round(n * frac_train))
    c2 = int(round(n * (frac_train + frac_val)))
    return idx_array[:c1], idx_array[c1:c2], idx_array[c2:]

wb_train, wb_val, wb_test = chrono_split(wet_before_idx)
wa_train, wa_val, wa_test = chrono_split(wet_after_idx)

train_idx = np.sort(np.concatenate([wb_train, dry_train_idx, wa_train]))
val_idx = np.sort(np.concatenate([wb_val, dry_val_idx, wa_val]))
test_idx = np.sort(np.concatenate([wb_test, dry_test_idx, wa_test]))

assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0
assert len(train_idx) + len(val_idx) + len(test_idx) == N

X_train, Y_train_abs, dates_train = X.iloc[train_idx].reset_index(drop=True), Y.iloc[train_idx].reset_index(drop=True), dates.iloc[train_idx].reset_index(drop=True)
X_val, Y_val_abs, dates_val = X.iloc[val_idx].reset_index(drop=True), Y.iloc[val_idx].reset_index(drop=True), dates.iloc[val_idx].reset_index(drop=True)
X_test, Y_test_abs, dates_test = X.iloc[test_idx].reset_index(drop=True), Y.iloc[test_idx].reset_index(drop=True), dates.iloc[test_idx].reset_index(drop=True)

print(f"Split sizes: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")

base_test = X_test[QIN_COL].to_numpy()

print("Zero-inflow rate by split AFTER stratification:")
for h, tcol in enumerate(targets, start=1):
    n_zero_tr = int((Y_train_abs[tcol] <= ZERO_THRESHOLD).sum())
    n_zero_val = int((Y_val_abs[tcol] <= ZERO_THRESHOLD).sum())
    n_zero_te = int((Y_test_abs[tcol] <= ZERO_THRESHOLD).sum())
    print(f"  H{h}: train {n_zero_tr}/{len(Y_train_abs)} ({100*n_zero_tr/len(Y_train_abs):.1f}%) | "
          f"val {n_zero_val}/{len(Y_val_abs)} ({100*n_zero_val/len(Y_val_abs):.1f}%) | "
          f"test {n_zero_te}/{len(Y_test_abs)} ({100*n_zero_te/len(Y_test_abs):.1f}%)")

print("\nNOTE: train/val/test are each built from THREE chronological pieces "
      "(early-wet + a dry-season slice + late-wet, where available). Train's "
      "zero-rate stays somewhat below val/test's -- this is a known, "
      "mathematically-forced limitation (train absorbs all 221 zero-free "
      "wet_before days), not a bug. Also: this test set's date range and "
      "composition differ from the chronological-split notebook's test set, "
      "so RMSE values are NOT directly comparable across the two notebooks "
      "-- only Model-vs-Baseline comparisons WITHIN each notebook are valid.")

Segments: wet_before=221 days | dry_window=104 days (45 zero) | wet_after=34 days
Dry window allocated proportionally: train=73, val=16, test=15
Split sizes: train=252, val=54, test=53
Zero-inflow rate by split AFTER stratification:
  H1: train 31/252 (12.3%) | val 10/54 (18.5%) | test 6/53 (11.3%)
  H2: train 31/252 (12.3%) | val 10/54 (18.5%) | test 6/53 (11.3%)
  H3: train 31/252 (12.3%) | val 10/54 (18.5%) | test 6/53 (11.3%)
  H4: train 31/252 (12.3%) | val 11/54 (20.4%) | test 5/53 (9.4%)
  H5: train 32/252 (12.7%) | val 10/54 (18.5%) | test 5/53 (9.4%)
  H6: train 32/252 (12.7%) | val 10/54 (18.5%) | test 5/53 (9.4%)
  H7: train 33/252 (13.1%) | val 9/54 (16.7%) | test 5/53 (9.4%)

NOTE: train/val/test are each built from THREE chronological pieces (early-wet + a dry-season slice + late-wet, where available). Train's zero-rate stays somewhat below val/test's -- this is a known, mathematically-forced limitation (train absorbs all 221 zero-free wet_before days), not a bug. Also: t

### Metric functions + persistence baseline

In [5]:
def mae_np(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat); return float(np.mean(np.abs(y - yhat)))

def rmse_np(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat); return float(np.sqrt(np.mean((y - yhat) ** 2)))

def nse_np(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    return float(1 - np.sum((y - yhat) ** 2) / (np.sum((y - y.mean()) ** 2) + 1e-9))

def pbias_np(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    return float(100 * np.sum(yhat - y) / (np.sum(y) + 1e-9))

def kge_np(y, yhat):
    """Kling-Gupta Efficiency."""
    y, yhat = np.asarray(y), np.asarray(yhat)
    r = np.corrcoef(y, yhat)[0, 1]
    alpha = (yhat.std() + 1e-9) / (y.std() + 1e-9)
    beta = (yhat.mean() + 1e-9) / (y.mean() + 1e-9)
    return float(1 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2))

baseline_rows = []
for h, tcol in enumerate(targets, start=1):
    y_true = Y_test_abs[tcol].to_numpy()
    baseline_rows.append({
        "H": h, "Target": tcol,
        "Baseline_MAE": mae_np(y_true, base_test), "Baseline_RMSE": rmse_np(y_true, base_test),
        "Baseline_NSE": nse_np(y_true, base_test), "Baseline_KGE": kge_np(y_true, base_test),
        "Baseline_PBIAS_%": pbias_np(y_true, base_test),
    })
baseline_df = pd.DataFrame(baseline_rows)
print("Persistence baseline (held-out test):")
print(baseline_df.to_string(index=False))

Persistence baseline (held-out test):
 H               Target  Baseline_MAE  Baseline_RMSE  Baseline_NSE  Baseline_KGE  Baseline_PBIAS_%
 1 y1=Q_in_t+1 (m3/day)   8625.630502   15492.320588      0.816658      0.765108         12.639045
 2 y2=Q_in_t+2 (m3/day)  13092.189195   22102.561778      0.460269      0.484546         26.280033
 3 y3=Q_in_t+3 (m3/day)  18375.281666   30130.401651     -0.041620      0.406719         26.613408
 4 y4=Q_in_t+4 (m3/day)  22714.795553   36098.581901     -0.605042      0.279666         27.814699
 5 y5=Q_in_t+5 (m3/day)  26636.648381   43105.348584     -0.702132      0.290226         20.688486
 6 y6=Q_in_t+6 (m3/day)  30905.915064   50864.519134     -0.603901      0.245571         10.704129
 7 y7=Q_in_t+7 (m3/day)  34390.162525   60391.231379     -0.446751      0.155582         -1.908930


## Part 2 — Optuna search per model per horizon (train-block-only inner CV)

ทำ Bayesian search แยกสำหรับ **ทั้ง 3 โมเดล x 7 horizons** (21 studies รวม) — search บน train block เท่านั้น ไม่แตะ val/test เลยจนกว่าจะเลือก best params ได้ จากนั้น refit ด้วย early stopping บน val แล้ว score test ครั้งเดียว


In [6]:
N_TRIALS = 40
INNER_SPLITS = 4

def make_objective(model_name, X_tr, y_tr_delta, qin_tr):
    tscv_inner = TimeSeriesSplit(n_splits=INNER_SPLITS)
    adapter = MODEL_ADAPTERS[model_name]
    def objective(trial):
        params = make_optuna_params(trial, model_name)
        fold_rmses = []
        for tr_idx, te_idx in tscv_inner.split(X_tr):
            Xtr_f, Xte_f = X_tr.iloc[tr_idx], X_tr.iloc[te_idx]
            ytr_f, yte_f = y_tr_delta[tr_idx], y_tr_delta[te_idx]
            qin_te_f = qin_tr[te_idx]
            m = adapter.build(params, with_early_stopping=False)
            m, _ = adapter.fit(m, Xtr_f, ytr_f)
            y_pred_abs_f = np.clip(qin_te_f + adapter.predict(m, Xte_f), 0, None)
            y_true_abs_f = yte_f + qin_te_f
            fold_rmses.append(rmse_np(y_true_abs_f, y_pred_abs_f))
        return float(np.mean(fold_rmses))
    return objective

In [7]:
all_test_results = []
all_tuned_params = []
fitted_models = {}  # (model_name, tcol) -> fitted model (for SHAP later)

for model_name, adapter in MODEL_ADAPTERS.items():
    print(f"=== {model_name} ===")
    for h, tcol in enumerate(targets, start=1):
        qin_train_arr = X_train[QIN_COL].to_numpy()
        y_train_delta = Y_train_abs[tcol].to_numpy() - qin_train_arr

        study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(make_objective(model_name, X_train, y_train_delta, qin_train_arr), n_trials=N_TRIALS)
        best_params = study.best_params
        all_tuned_params.append({"Model": model_name, "H": h, "Target": tcol,
                                  "inner_cv_rmse": study.best_value, **best_params})

        yval_delta = Y_val_abs[tcol].to_numpy() - X_val[QIN_COL].to_numpy()
        m = adapter.build(best_params, with_early_stopping=True)
        m, best_iter = adapter.fit(m, X_train, y_train_delta, X_val, yval_delta)
        fitted_models[(model_name, tcol)] = m

        delta_pred_test = adapter.predict(m, X_test)
        q_pred_test = np.clip(base_test + delta_pred_test, 0, None)
        y_true_test = Y_test_abs[tcol].to_numpy()

        delta_pred_val = adapter.predict(m, X_val)
        q_pred_val = np.clip(X_val[QIN_COL].to_numpy() + delta_pred_val, 0, None)

        all_test_results.append({
            "Model": model_name, "H": h, "Target": tcol, "best_iter": best_iter,
            "inner_cv_rmse": study.best_value,
            "Baseline_MAE": mae_np(y_true_test, base_test), "Test_MAE": mae_np(y_true_test, q_pred_test),
            "MAE_improve_%": 100 * (mae_np(y_true_test, base_test) - mae_np(y_true_test, q_pred_test)) / (mae_np(y_true_test, base_test) + 1e-9),
            "Baseline_RMSE": rmse_np(y_true_test, base_test), "Test_RMSE": rmse_np(y_true_test, q_pred_test),
            "RMSE_improve_%": 100 * (rmse_np(y_true_test, base_test) - rmse_np(y_true_test, q_pred_test)) / (rmse_np(y_true_test, base_test) + 1e-9),
            "Baseline_NSE": nse_np(y_true_test, base_test), "Test_NSE": nse_np(y_true_test, q_pred_test),
            "Test_KGE": kge_np(y_true_test, q_pred_test),
            "Val_NSE": nse_np(Y_val_abs[tcol].to_numpy(), q_pred_val),
        })
        print(f"  H{h}: Test_NSE={nse_np(y_true_test, q_pred_test):.4f} (baseline={nse_np(y_true_test, base_test):.4f}), "
              f"Val_NSE={nse_np(Y_val_abs[tcol].to_numpy(), q_pred_val):.4f}")

results_df = pd.DataFrame(all_test_results)
tuned_params_df = pd.DataFrame(all_tuned_params)
results_df.to_csv(os.path.join(OUTDIR, "all_models_test_performance.csv"), index=False)
tuned_params_df.to_csv(os.path.join(OUTDIR, "all_models_tuned_params.csv"), index=False)

=== CatBoost ===
  H1: Test_NSE=0.8366 (baseline=0.8167), Val_NSE=0.9861
  H2: Test_NSE=0.6096 (baseline=0.4603), Val_NSE=0.9795
  H3: Test_NSE=0.1716 (baseline=-0.0416), Val_NSE=0.9729
  H4: Test_NSE=-0.3069 (baseline=-0.6050), Val_NSE=0.9146
  H5: Test_NSE=-0.7021 (baseline=-0.7021), Val_NSE=0.7438
  H6: Test_NSE=-0.5617 (baseline=-0.6039), Val_NSE=0.5702
  H7: Test_NSE=-0.3881 (baseline=-0.4468), Val_NSE=0.6211
=== XGBoost ===
  H1: Test_NSE=0.8017 (baseline=0.8167), Val_NSE=0.9789
  H2: Test_NSE=0.3829 (baseline=0.4603), Val_NSE=0.9399
  H3: Test_NSE=-0.1632 (baseline=-0.0416), Val_NSE=0.8701
  H4: Test_NSE=-0.7857 (baseline=-0.6050), Val_NSE=0.7430
  H5: Test_NSE=-0.8434 (baseline=-0.7021), Val_NSE=0.5556
  H6: Test_NSE=-0.6861 (baseline=-0.6039), Val_NSE=0.2982
  H7: Test_NSE=-0.4764 (baseline=-0.4468), Val_NSE=-0.0122
=== LightGBM ===
  H1: Test_NSE=0.7961 (baseline=0.8167), Val_NSE=0.9775
  H2: Test_NSE=0.3849 (baseline=0.4603), Val_NSE=0.9406
  H3: Test_NSE=-0.1586 (baseline=-

## Part 3 — Walk-forward CV diagnostic (train block only) for each model

ใช้ best params ที่ได้จาก Optuna ของแต่ละโมเดล ตรวจความเสถียรด้วย walk-forward CV บน train block เท่านั้น


In [8]:
tscv = TimeSeriesSplit(n_splits=5)
cv_rows = []

for model_name, adapter in MODEL_ADAPTERS.items():
    for fold, (tr, te) in enumerate(tscv.split(X_train), start=1):
        Xtr_cv, Xte_cv = X_train.iloc[tr].reset_index(drop=True), X_train.iloc[te].reset_index(drop=True)
        base_te_cv = Xte_cv[QIN_COL].to_numpy()
        for h, tcol in enumerate(targets, start=1):
            ytr_abs_cv = Y_train_abs[tcol].iloc[tr].reset_index(drop=True).to_numpy()
            yte_abs_cv = Y_train_abs[tcol].iloc[te].reset_index(drop=True).to_numpy()
            ytr_delta_cv = ytr_abs_cv - Xtr_cv[QIN_COL].to_numpy()

            tuned_row = tuned_params_df[(tuned_params_df["Model"] == model_name) & (tuned_params_df["H"] == h)].iloc[0]
            param_cols = [c for c in tuned_params_df.columns if c not in ("Model", "H", "Target", "inner_cv_rmse")]
            params = {c: tuned_row[c] for c in param_cols if pd.notna(tuned_row[c])}
            # cast int-like params back to int (Optuna ints survive CSV roundtrip as floats)
            for k in ["iterations", "n_estimators", "depth", "max_depth", "min_data_in_leaf",
                      "min_child_weight", "min_child_samples", "subsample_freq"]:
                if k in params:
                    params[k] = int(params[k])

            m_cv = adapter.build(params, with_early_stopping=False)
            m_cv, _ = adapter.fit(m_cv, Xtr_cv, ytr_delta_cv)
            yhat_cv = np.clip(base_te_cv + adapter.predict(m_cv, Xte_cv), 0, None)

            cv_rows.append({
                "Model": model_name, "fold": fold, "H": h,
                "Baseline_NSE": nse_np(yte_abs_cv, base_te_cv),
                "Model_NSE": nse_np(yte_abs_cv, yhat_cv),
            })

cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv(os.path.join(OUTDIR, "cv_stability_diagnostic.csv"), index=False)
print("Walk-forward CV stability (train block only), mean +/- std across folds, per model per horizon:")
print(cv_df.groupby(["Model", "H"])[["Model_NSE"]].agg(["mean", "std"]))

Walk-forward CV stability (train block only), mean +/- std across folds, per model per horizon:
            Model_NSE           
                 mean        std
Model    H                      
CatBoost 1   0.503636   0.423660
         2   0.350534   0.277672
         3   0.029369   0.449367
         4  -0.194842   0.445026
         5  -0.410459   0.586249
         6  -0.691140   0.809525
         7  -0.714310   0.746801
LightGBM 1  -0.561361   1.935561
         2  -0.113268   0.762632
         3  -3.257250   4.664972
         4  -2.361688   3.242128
         5  -2.638224   4.142505
         6  -3.644822   5.322783
         7 -12.255130  21.784726
XGBoost  1   0.420954   0.612711
         2  -0.025054   0.670684
         3  -0.559616   1.226893
         4  -1.817526   2.530903
         5  -2.596951   3.620716
         6  -9.111802  14.983896
         7  -2.145476   3.093798


## Part 4 — SHAP feature importance per model

ใช้ test set เดียวกันสำหรับทุกโมเดล เพื่อเทียบ feature importance ได้ตรงกัน


In [9]:
shap_importance_all = []
for model_name in MODEL_ADAPTERS:
    for tcol in targets:
        m = fitted_models[(model_name, tcol)]
        explainer = shap.TreeExplainer(m)
        sv = explainer.shap_values(X_test)
        mean_abs_shap = np.abs(sv).mean(axis=0)
        for feat, val in zip(X_test.columns, mean_abs_shap):
            shap_importance_all.append({"Model": model_name, "Target": tcol, "feature": feat, "mean_abs_shap": float(val)})

shap_df = pd.DataFrame(shap_importance_all)
shap_df.to_csv(os.path.join(OUTDIR, "shap_importance_all_models.csv"), index=False)

print("Top SHAP feature per model, H1:")
for model_name in MODEL_ADAPTERS:
    sub = shap_df[(shap_df["Model"] == model_name) & (shap_df["Target"] == targets[0])].sort_values("mean_abs_shap", ascending=False)
    print(f"  {model_name}: {sub.iloc[0]['feature']} ({sub.iloc[0]['mean_abs_shap']:.1f})")

Top SHAP feature per model, H1:
  CatBoost: Water_Level_t (m) (4109.5)
  XGBoost: API_t (mm) (124.8)
  LightGBM: Q_in_t (m3/day) (421.1)


## Part 5 — Cross-model comparison: ตัวไหนดีที่สุด?

สรุปผลทั้ง 3 โมเดลเทียบกัน เพื่อตอบคำถามว่าควรเปลี่ยนจาก CatBoost ไปใช้โมเดลอื่นหรือไม่


In [10]:
print("=== Test_NSE by Model x Horizon ===")
pivot_nse = results_df.pivot_table(index="H", columns="Model", values="Test_NSE")
print(pivot_nse.to_string())

print()
print("=== Test_RMSE by Model x Horizon (lower is better) ===")
pivot_rmse = results_df.pivot_table(index="H", columns="Model", values="Test_RMSE")
print(pivot_rmse.to_string())

print()
print("=== Test_MAE by Model x Horizon (lower is better) ===")
pivot_mae = results_df.pivot_table(index="H", columns="Model", values="Test_MAE")
print(pivot_mae.to_string())

print()
print("=== Best model per horizon (by Test_NSE) ===")
best_per_h = results_df.loc[results_df.groupby("H")["Test_NSE"].idxmax(), ["H", "Model", "Test_NSE"]]
print(best_per_h.to_string(index=False))

print()
print("=== Win count across 7 horizons ===")
print(best_per_h["Model"].value_counts().to_string())

print()
print("=== Average Test_NSE across all horizons (overall ranking) ===")
print(results_df.groupby("Model")["Test_NSE"].mean().sort_values(ascending=False).to_string())

print()
print("=== Average Test_MAE across all horizons (overall ranking, lower is better) ===")
print(results_df.groupby("Model")["Test_MAE"].mean().sort_values().to_string())

=== Test_NSE by Model x Horizon ===
Model  CatBoost  LightGBM   XGBoost
H                                  
1      0.836645  0.796096  0.801740
2      0.609584  0.384861  0.382937
3      0.171574 -0.158571 -0.163186
4     -0.306875 -0.785450 -0.785750
5     -0.702087 -0.834802 -0.843434
6     -0.561728 -0.677350 -0.686124
7     -0.388141 -0.468987 -0.476360

=== Test_RMSE by Model x Horizon (lower is better) ===
Model      CatBoost      LightGBM       XGBoost
H                                              
1      14623.489724  16337.955282  16110.252213
2      18798.244154  23596.107510  23632.977961
3      26870.589160  31776.904323  31840.127748
4      32573.460647  38073.323281  38076.512079
5      43104.777448  44753.712912  44858.872548
6      50191.342128  52016.132502  52151.993807
7      59155.311480  60853.560965  61006.088026

=== Test_MAE by Model x Horizon (lower is better) ===
Model      CatBoost      LightGBM       XGBoost
H                                              
1

### วิธีอ่านผลลัพธ์

ดูที่ **"Win count"** และ **"Average Test_NSE"** ด้านบนเพื่อตัดสินว่าโมเดลไหนดีที่สุดโดยรวมใน split นี้ จากนั้นเทียบกับผลใน notebook อีกไฟล์ (chronological split) เพื่อดู 2 อย่าง:

1. **`Val_NSE` ดีขึ้นทุกโมเดลหรือไม่** — ถ้าใช่ (ตามที่พบในรอบทดสอบ CatBoost เดี่ยวก่อนหน้า) แสดงว่า ปัญหาเดิมที่เคยเจอเป็นปัญหาจาก split ไม่ใช่จากโมเดล และ early stopping/Optuna search ในไฟล์นี้ น่าเชื่อถือกว่าไฟล์เดิม
2. **โมเดลที่ชนะเปลี่ยนไปหรือไม่** — ถ้าโมเดลที่ชนะเปลี่ยนระหว่าง 2 split แสดงว่าการเลือกโมเดล "ดีที่สุด" ยังไม่เสถียรพอที่จะสรุปเป็นค่าตายตัว ควรพิจารณาทั้ง 2 ผลลัพธ์ร่วมกัน ไม่ใช่เชื่อไฟล์เดียว

**ข้อควรจำ:** ห้ามเทียบ RMSE ตัวเลขตรงๆข้าม 2 notebook เพราะ test set มีองค์ประกอบและ scale ต่างกัน — เทียบได้แค่ "Model ดีกว่า Baseline แค่ไหน" ภายในแต่ละไฟล์ แล้วดูว่าระดับการดีกว่านั้นสอดคล้องกันข้าม 2 split หรือไม่


---
## Two-Stage (Zero-Inflated / Hurdle) Model — โมดูลเสริม, รองรับ 3 โมเดล

**แนวคิด:** ข้อมูลนี้มี `Q_in_t = 0` กระจุกตัวในฤดูแล้ง (104 วันต่อกัน, 2026-02-03 ถึง 2026-05-17) แทนที่จะให้ regressor เดียวทำนายทั้ง "0 แบนราบ" กับ "เหตุการณ์น้ำมาก" ในฟังก์ชันเดียว ใช้โครงสร้าง 2 ขั้น:

- **Stage 1 (classifier):** ทำนาย P(Q_in_{t+h} = 0) — **ใช้ CatBoostClassifier เท่านั้น** (ดูเหตุผลด้านล่าง)
- **Stage 2 (regressor):** เทรนเฉพาะแถวที่ Q_in_actual > 0 ใน train เท่านั้น — **รัน 3 โมเดล (CatBoost, XGBoost, LightGBM) พร้อม Optuna search แยกต่อโมเดลต่อ horizon** เพื่อตอบคำถามว่าโมเดลไหนเป็น Stage-2 regressor ที่ดีที่สุด
- **Final prediction:** ถ้า Stage 1 บอกว่า "0" (prob >= threshold) → 0, ไม่งั้น → Stage 2 ของโมเดลนั้นๆ

> **ทำไม Stage 1 ใช้ CatBoost ตัวเดียว ไม่ทำ 3 โมเดลเหมือน Stage 2:** จากการทดสอบโมดูลนี้เวอร์ชันเดี่ยวก่อนหน้า พบว่าคอขวดของ Stage 1 คือ **จำนวนตัวอย่าง zero-inflow ในtrain มีแค่ 6-10 แถวต่อ horizon** ทำให้การเลือก threshold มี noise สูง ไม่ใช่ปัญหาจาก algorithm ของ classifier เอง การทำ fix เดิม (fixed iterations + stratified threshold resampling) ซ้ำสำหรับ XGBoost/LightGBM classifier จะเพิ่มความซับซ้อนโดยไม่แก้ปัญหาที่ต้นเหตุ จึงให้ Stage 1 เป็น CatBoost ตัวเดียว และให้ Stage 2 (จุดที่ "เลือกโมเดลไหนดีกว่า" มีความหมายจริง) รองรับ 3 โมเดลแทน

**Leakage discipline:** เหมือนเดิมทุกประการ — filter ">0" ทำใน X_train เท่านั้น, threshold เลือกจาก X_val เท่านั้น, X_test ถูกแตะครั้งเดียวต่อโมเดล Stage 2

> **Stage 2 ใช้ fixed iterations (อัปเดตจากเวอร์ชันก่อน):** เดิม Stage 2 ใช้ early stopping กับ X_val เต็มชุด ซึ่งพบว่าพังจริง — Stage 2 เทรนเฉพาะแถว Q_in_actual>0 แต่ X_val มีวัน zero-inflow ปนอยู่ ทำให้ early stopping เก็บ iteration แรกสุดบ่อยๆ (โมเดลไม่เรียนรู้อะไรเลย) ตอนนี้เปลี่ยนเป็น fixed iterations (จาก Optuna) พร้อมรายงาน in-train CV (`in_train_cv_nse`) ไว้ตรงๆให้ตรวจสอบเอง — ดูรายละเอียดเหตุผลที่ไม่ใช้ automatic fallback ในส่วน "วิธีอ่านผลลัพธ์" ด้านล่าง


In [11]:
print("=" * 78)
print(" TWO-STAGE (HURDLE) MODEL -- Stage 1: CatBoost classifier | Stage 2: 3 models")
print("=" * 78)

from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve
from sklearn.model_selection import StratifiedKFold

ZERO_THRESHOLD = 1e-6

print("Zero-inflow rate by split (diagnostic):")
for h, tcol in enumerate(targets, start=1):
    n_zero_tr = int((Y_train_abs[tcol] <= ZERO_THRESHOLD).sum())
    n_zero_val = int((Y_val_abs[tcol] <= ZERO_THRESHOLD).sum())
    n_zero_te = int((Y_test_abs[tcol] <= ZERO_THRESHOLD).sum())
    print(f"  H{h}: train {n_zero_tr}/{len(Y_train_abs)} ({100*n_zero_tr/len(Y_train_abs):.1f}%) | "
          f"val {n_zero_val}/{len(Y_val_abs)} ({100*n_zero_val/len(Y_val_abs):.1f}%) | "
          f"test {n_zero_te}/{len(Y_test_abs)} ({100*n_zero_te/len(Y_test_abs):.1f}%)")

 TWO-STAGE (HURDLE) MODEL -- Stage 1: CatBoost classifier | Stage 2: 3 models
Zero-inflow rate by split (diagnostic):
  H1: train 31/252 (12.3%) | val 10/54 (18.5%) | test 6/53 (11.3%)
  H2: train 31/252 (12.3%) | val 10/54 (18.5%) | test 6/53 (11.3%)
  H3: train 31/252 (12.3%) | val 10/54 (18.5%) | test 6/53 (11.3%)
  H4: train 31/252 (12.3%) | val 11/54 (20.4%) | test 5/53 (9.4%)
  H5: train 32/252 (12.7%) | val 10/54 (18.5%) | test 5/53 (9.4%)
  H6: train 32/252 (12.7%) | val 10/54 (18.5%) | test 5/53 (9.4%)
  H7: train 33/252 (13.1%) | val 9/54 (16.7%) | test 5/53 (9.4%)


### Stage 1 — CatBoost Classifier (fixed iterations, threshold เลือกจาก stratified resample ของ val)


In [12]:
# =============================================================================
# Stage 1 -- CatBoost Classifier: P(Q_in_{t+h} = 0)
# Fixed iterations (no early stopping against val -- val's zero-rate can be
# very different from train's, which previously caused early stopping to
# collapse to iteration 0). Quality checked via in-train stratified CV;
# deployment threshold chosen on a stratified resample of val matching
# train's zero-rate.
# =============================================================================
clf_params = dict(
    loss_function="Logloss", iterations=200, learning_rate=0.05, depth=3,
    l2_leaf_reg=10, auto_class_weights="Balanced", random_seed=42, verbose=False,
)

stage1_models = {}
stage1_thresholds = {}
stage1_diag_rows = []

for h, tcol in enumerate(targets, start=1):
    is_zero_train = (Y_train_abs[tcol].to_numpy() <= ZERO_THRESHOLD).astype(int)
    is_zero_val = (Y_val_abs[tcol].to_numpy() <= ZERO_THRESHOLD).astype(int)

    if is_zero_train.sum() < 3 or is_zero_train.sum() > len(is_zero_train) - 3:
        print(f"H{h}: too few zero/non-zero examples in TRAIN ({is_zero_train.sum()}) -> Stage 1 skipped for this horizon.")
        stage1_models[tcol] = None
        stage1_thresholds[tcol] = None
        continue

    skf = StratifiedKFold(n_splits=min(5, int(is_zero_train.sum())), shuffle=True, random_state=42)
    cv_aucs = []
    for tr_i, te_i in skf.split(X_train, is_zero_train):
        clf_fold = CatBoostClassifier(**clf_params)
        clf_fold.fit(X_train.iloc[tr_i], is_zero_train[tr_i], verbose=False)
        p_fold = clf_fold.predict_proba(X_train.iloc[te_i])[:, 1]
        if 0 < is_zero_train[te_i].sum() < len(te_i):
            cv_aucs.append(roc_auc_score(is_zero_train[te_i], p_fold))
    in_train_cv_auc = float(np.mean(cv_aucs)) if cv_aucs else float("nan")

    clf = CatBoostClassifier(**clf_params)
    clf.fit(X_train, is_zero_train, verbose=False)
    stage1_models[tcol] = clf
    val_proba_zero = clf.predict_proba(X_val)[:, 1]

    rng = np.random.RandomState(42)
    zero_idx_val = np.where(is_zero_val == 1)[0]
    nonzero_idx_val = np.where(is_zero_val == 0)[0]
    train_zero_rate = is_zero_train.mean()

    if len(zero_idx_val) >= 2 and len(nonzero_idx_val) >= 2:
        n_zero_keep = len(zero_idx_val)
        n_nonzero_target = int(round(n_zero_keep * (1 - train_zero_rate) / max(train_zero_rate, 1e-6)))
        n_nonzero_keep = min(len(nonzero_idx_val), max(n_nonzero_target, 2))
        nonzero_sample = rng.choice(nonzero_idx_val, size=n_nonzero_keep, replace=(n_nonzero_keep > len(nonzero_idx_val)))
        strat_idx = np.concatenate([zero_idx_val, nonzero_sample])
        y_strat, p_strat = is_zero_val[strat_idx], val_proba_zero[strat_idx]
    else:
        y_strat, p_strat = is_zero_val, val_proba_zero

    if 0 < y_strat.sum() < len(y_strat):
        prec, rec, thr = precision_recall_curve(y_strat, p_strat)
        f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(prec), where=(prec + rec) > 0)
        best_idx = np.argmax(f1[:-1]) if len(thr) > 0 else 0
        best_thr = float(thr[best_idx]) if len(thr) > 0 else 0.5
        val_auc = roc_auc_score(is_zero_val, val_proba_zero)
    else:
        best_thr, val_auc = 0.5, float("nan")

    stage1_thresholds[tcol] = best_thr
    stage1_diag_rows.append({
        "H": h, "n_zero_train": int(is_zero_train.sum()), "n_zero_val": int(is_zero_val.sum()),
        "in_train_cv_auc": in_train_cv_auc, "val_auc": val_auc, "chosen_threshold": best_thr,
    })

stage1_diag_df = pd.DataFrame(stage1_diag_rows)
print("Stage 1 diagnostics (in_train_cv_auc is the honest signal; val_auc is unreliable under split mismatch):")
print(stage1_diag_df.to_string(index=False))

Stage 1 diagnostics (in_train_cv_auc is the honest signal; val_auc is unreliable under split mismatch):
 H  n_zero_train  n_zero_val  in_train_cv_auc  val_auc  chosen_threshold
 1            31          10         0.953997 0.943182          0.566280
 2            31          10         0.944548 0.934091          0.703928
 3            31          10         0.943870 0.932955          0.536321
 4            31          11         0.953228 0.955603          0.696930
 5            32          10         0.933225 0.912500          0.359991
 6            32          10         0.934470 0.888636          0.319231
 7            33           9         0.934859 0.906173          0.569049


### Stage 2 — Regressor 3 โมเดล (Optuna search แยกต่อโมเดลต่อ horizon, เทรนเฉพาะแถว Q_in_actual > 0 ใน train)


In [13]:
# =============================================================================
# Stage 2 -- Regressor (3 models), trained ONLY on Q_in_actual > 0 rows in TRAIN
#
# FIX (this version): Stage 2 now uses FIXED iterations (no early stopping
# against X_val), for the same reason Stage 1's classifier was fixed earlier.
#
# Root cause confirmed by direct reproduction: Stage 2 is trained only on
# Q_in_actual > 0 rows, but early stopping was evaluating against the FULL
# X_val (including zero-inflow days the model was never trained to predict).
# As soon as the model learned anything from the positive-only training
# distribution, its loss on the mixed-population val set got WORSE, not
# better -- val RMSE increased monotonically from round 0 onward in the
# reproduced case (XGBoost, H5: 45,574 -> 45,775 -> ... -> 47,689 by round 9).
# This made early stopping keep iteration 0 -- a model that learned nothing
# beyond the global mean delta -- for XGBoost at H4-H7 and intermittently
# for LightGBM, in a pattern structurally identical to Stage 1's earlier
# best_iteration=0 collapse.
#
# Each model still gets its own Optuna search (train-block-only inner CV,
# same protocol as Part 2 above) to choose hyperparameters INCLUDING the
# iteration budget (n_estimators / iterations is itself a tuned parameter).
# The final fit uses that exact budget with no early stopping. Quality is
# checked via an in-train CV diagnostic (analogous to Stage 1's
# in_train_cv_auc) rather than trusting a val-based stopping signal that
# reflects the wrong population.
# =============================================================================
N_TRIALS_STAGE2 = 25  # somewhat fewer than Part 2's 40, since Stage 2 trains
                       # on an even smaller positive-only subset of train
INNER_SPLITS_STAGE2 = 3

def make_stage2_objective(model_name, X_tr_pos, y_tr_pos_delta):
    adapter = MODEL_ADAPTERS[model_name]
    tscv_inner = TimeSeriesSplit(n_splits=min(INNER_SPLITS_STAGE2, max(2, len(X_tr_pos) // 20)))
    def objective(trial):
        params = make_optuna_params(trial, model_name)
        fold_rmses = []
        for tr_idx, te_idx in tscv_inner.split(X_tr_pos):
            Xtr_f, Xte_f = X_tr_pos.iloc[tr_idx], X_tr_pos.iloc[te_idx]
            ytr_f, yte_f = y_tr_pos_delta[tr_idx], y_tr_pos_delta[te_idx]
            qin_te_f = Xte_f[QIN_COL].to_numpy()
            # Fixed-iteration fit during search too, so the search optimises
            # the SAME training protocol that will be used for the final fit
            # (searching with early stopping enabled, then deploying without
            # it, would let the search implicitly tune against a stopping
            # signal the final model never gets to use).
            m = adapter.build(params, with_early_stopping=False)
            m, _ = adapter.fit(m, Xtr_f, ytr_f)
            y_pred_abs_f = np.clip(qin_te_f + adapter.predict(m, Xte_f), 0, None)
            y_true_abs_f = yte_f + qin_te_f
            fold_rmses.append(rmse_np(y_true_abs_f, y_pred_abs_f))
        return float(np.mean(fold_rmses))
    return objective

stage2_models = {}       # (model_name, tcol) -> fitted model
stage2_tuned_params = []
stage2_diag_rows = []    # in-train CV diagnostic, analogous to Stage 1's in_train_cv_auc

for model_name, adapter in MODEL_ADAPTERS.items():
    print(f"--- Stage 2: {model_name} ---")
    for h, tcol in enumerate(targets, start=1):
        pos_mask_train = Y_train_abs[tcol].to_numpy() > ZERO_THRESHOLD
        X_train_pos = X_train.loc[pos_mask_train].reset_index(drop=True)
        qin_train_pos = X_train_pos[QIN_COL].to_numpy()
        y_train_pos_delta = Y_train_abs[tcol].to_numpy()[pos_mask_train] - qin_train_pos

        study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(make_stage2_objective(model_name, X_train_pos, y_train_pos_delta), n_trials=N_TRIALS_STAGE2)
        best_params = study.best_params
        stage2_tuned_params.append({"Model": model_name, "H": h, "inner_cv_rmse": study.best_value, **best_params})

        # Final fit: FIXED iterations from the tuned params, no early stopping.
        m2 = adapter.build(best_params, with_early_stopping=False)
        m2, n_iters_used = adapter.fit(m2, X_train_pos, y_train_pos_delta)
        stage2_models[(model_name, tcol)] = m2

        # In-train CV diagnostic: repeat the inner-CV scoring at the chosen
        # params so there is an honest, val-independent quality signal to
        # report alongside the deployment model (mirrors Stage 1's
        # in_train_cv_auc, which served the same role for the classifier).
        diag_objective = make_stage2_objective(model_name, X_train_pos, y_train_pos_delta)
        # Re-score the exact chosen params (not a new search) by calling the
        # same fold-loop logic directly:
        tscv_diag = TimeSeriesSplit(n_splits=min(INNER_SPLITS_STAGE2, max(2, len(X_train_pos) // 20)))
        diag_rmses = []
        for tr_idx, te_idx in tscv_diag.split(X_train_pos):
            Xtr_f, Xte_f = X_train_pos.iloc[tr_idx], X_train_pos.iloc[te_idx]
            ytr_f, yte_f = y_train_pos_delta[tr_idx], y_train_pos_delta[te_idx]
            qin_te_f = Xte_f[QIN_COL].to_numpy()
            m_diag = adapter.build(best_params, with_early_stopping=False)
            m_diag, _ = adapter.fit(m_diag, Xtr_f, ytr_f)
            y_pred_abs_f = np.clip(qin_te_f + adapter.predict(m_diag, Xte_f), 0, None)
            y_true_abs_f = yte_f + qin_te_f
            diag_rmses.append(nse_np(y_true_abs_f, y_pred_abs_f))
        in_train_cv_nse = float(np.mean(diag_rmses))
        stage2_diag_rows.append({"Model": model_name, "H": h, "in_train_cv_nse": in_train_cv_nse})

        print(f"  H{h}: trained on {pos_mask_train.sum()}/{len(pos_mask_train)} TRAIN rows (Q_in_actual>0), "
              f"inner_cv_rmse={study.best_value:.0f}, n_iters_used={n_iters_used}, "
              f"in_train_cv_NSE={in_train_cv_nse:.3f}")

stage2_params_df = pd.DataFrame(stage2_tuned_params)
stage2_params_df.to_csv(os.path.join(OUTDIR, "hurdle_stage2_tuned_params.csv"), index=False)

stage2_diag_df = pd.DataFrame(stage2_diag_rows)
stage2_diag_df.to_csv(os.path.join(OUTDIR, "hurdle_stage2_intrain_cv_diagnostic.csv"), index=False)
print("\nStage 2 in-train CV diagnostic (honest, val-independent quality signal):")
print(stage2_diag_df.pivot_table(index="H", columns="Model", values="in_train_cv_nse").to_string())

# =============================================================================
# NOTE on what was tried and rejected here, kept for the record because it
# matters for how to interpret the results below:
#
# An automatic safety net was attempted, in two forms, to catch Stage 2 fits
# that fail to generalize once early stopping was removed:
#   (1) auto-fallback to persistence when in_train_cv_nse < -1.0
#   (2) a per-row "extrapolation ceiling" (flag predictions far outside the
#       range of values seen in training, replace with persistence)
# Both were rejected after direct testing:
#   (1) failed because in_train_cv_nse is not a reliable predictor of test
#       failure here -- XGBoost's in_train_cv_nse stayed in [-0.56, 0.75]
#       (never breaching the -1.0 trigger) while test NSE still collapsed
#       to -20.7 at H4. The ~220-row positive-only training set does not
#       cover enough of the conditions test will contain.
#   (2) failed because this dataset's training distribution is itself
#       contaminated by a handful of extreme flood-year rows (one row
#       reaches 1,408,228 against a typical/median positive value of
#       ~35,000-42,000), so any percentile- or max-based ceiling derived
#       from training data is too loose to ever trigger. A tighter,
#       robust-statistic-based ceiling (median x10, or 15x the current
#       day's persistence) also failed to catch the actual bad
#       predictions, because those predictions are not physically
#       implausible values -- they are ordinary-looking numbers that are
#       simply WRONG. A 2-4x jump from one day's flow to a few days later
#       is not an impossible number for this reservoir; the model is just
#       inaccurate at that horizon. No plausible-range filter can fix an
#       accuracy problem -- only more data or a different feature
#       set/model could.
#
# Conclusion: this is reported honestly rather than patched with a guard
# that does not address the real cause. in_train_cv_nse remains in the
# output as the most useful available signal -- read it directly per
# (model, horizon) rather than trusting any single automatic threshold.
# =============================================================================

# =============================================================================
# Combine Stage 1 + Stage 2 (per model), score TEST exactly once per model.
# =============================================================================
hurdle_test_rows = []
hurdle_pred_store = {}

for model_name, adapter in MODEL_ADAPTERS.items():
    for h, tcol in enumerate(targets, start=1):
        y_true_test = Y_test_abs[tcol].to_numpy()
        clf = stage1_models[tcol]
        thr = stage1_thresholds[tcol]
        m2 = stage2_models[(model_name, tcol)]
        in_train_cv_nse_this = next(r["in_train_cv_nse"] for r in stage2_diag_rows
                                     if r["Model"] == model_name and r["H"] == h)

        delta_pred_test = adapter.predict(m2, X_test)
        stage2_pred_test = np.clip(base_test + delta_pred_test, 0, None)

        if clf is not None:
            proba_zero_test = clf.predict_proba(X_test)[:, 1]
            is_pred_zero = proba_zero_test >= thr
        else:
            is_pred_zero = np.zeros(len(X_test), dtype=bool)

        final_pred_test = np.where(is_pred_zero, 0.0, stage2_pred_test)
        hurdle_pred_store[(model_name, tcol)] = final_pred_test

        actual_zero_test = y_true_test <= ZERO_THRESHOLD
        zero_recall = float((is_pred_zero & actual_zero_test).sum() / actual_zero_test.sum()) if actual_zero_test.sum() > 0 else float("nan")

        hurdle_test_rows.append({
            "Model": model_name, "H": h, "Target": tcol,
            "stage2_in_train_cv_nse": in_train_cv_nse_this,
            "n_actual_zero_test": int(actual_zero_test.sum()), "n_pred_zero_test": int(is_pred_zero.sum()),
            "zero_recall": zero_recall,
            "Baseline_MAE": mae_np(y_true_test, base_test),
            "SingleStage_MAE": mae_np(y_true_test, stage2_pred_test),
            "Hurdle_MAE": mae_np(y_true_test, final_pred_test),
            "Baseline_RMSE": rmse_np(y_true_test, base_test),
            "SingleStage_RMSE": rmse_np(y_true_test, stage2_pred_test),
            "Hurdle_RMSE": rmse_np(y_true_test, final_pred_test),
            "Baseline_NSE": nse_np(y_true_test, base_test),
            "SingleStage_NSE": nse_np(y_true_test, stage2_pred_test),
            "Hurdle_NSE": nse_np(y_true_test, final_pred_test),
        })

hurdle_test_df = pd.DataFrame(hurdle_test_rows)
hurdle_test_df.to_csv(os.path.join(OUTDIR, "hurdle_test_performance_all_models.csv"), index=False)
print("=== Two-stage (hurdle) model -- held-out TEST performance, all 3 Stage-2 models ===")
print(hurdle_test_df.to_string(index=False))

--- Stage 2: CatBoost ---
  H1: trained on 221/252 TRAIN rows (Q_in_actual>0), inner_cv_rmse=95317, n_iters_used=1200, in_train_cv_NSE=0.750
  H2: trained on 221/252 TRAIN rows (Q_in_actual>0), inner_cv_rmse=128194, n_iters_used=1300, in_train_cv_NSE=0.556
  H3: trained on 221/252 TRAIN rows (Q_in_actual>0), inner_cv_rmse=157762, n_iters_used=500, in_train_cv_NSE=0.307
  H4: trained on 221/252 TRAIN rows (Q_in_actual>0), inner_cv_rmse=180801, n_iters_used=300, in_train_cv_NSE=0.096
  H5: trained on 220/252 TRAIN rows (Q_in_actual>0), inner_cv_rmse=226081, n_iters_used=400, in_train_cv_NSE=-0.677
  H6: trained on 220/252 TRAIN rows (Q_in_actual>0), inner_cv_rmse=245932, n_iters_used=400, in_train_cv_NSE=-1.120
  H7: trained on 219/252 TRAIN rows (Q_in_actual>0), inner_cv_rmse=241334, n_iters_used=300, in_train_cv_NSE=-4.524
--- Stage 2: XGBoost ---
  H1: trained on 221/252 TRAIN rows (Q_in_actual>0), inner_cv_rmse=88700, n_iters_used=900, in_train_cv_NSE=0.769
  H2: trained on 221/252 T

### เปรียบเทียบ Hurdle ข้าม 3 โมเดล


In [14]:
# =============================================================================
# Hurdle vs Single-stage vs Baseline, compared across all 3 Stage-2 models
# =============================================================================
print("=== Hurdle_NSE by Model x Horizon ===")
print(hurdle_test_df.pivot_table(index="H", columns="Model", values="Hurdle_NSE").to_string())

print()
print("=== Stage 2 in_train_cv_nse by Model x Horizon (read this alongside Hurdle_NSE above --")
print("    a model with poor in_train_cv_nse at a given horizon should not be trusted there,")
print("    even if its test-set number happens to look acceptable) ===")
print(hurdle_test_df.pivot_table(index="H", columns="Model", values="stage2_in_train_cv_nse", aggfunc="first").to_string())

print()
print("=== Best Stage-2 model per horizon (by Hurdle_NSE) ===")
best_hurdle_per_h = hurdle_test_df.loc[hurdle_test_df.groupby("H")["Hurdle_NSE"].idxmax(), ["H", "Model", "Hurdle_NSE"]]
print(best_hurdle_per_h.to_string(index=False))

print()
print("=== Win count across 7 horizons (hurdle Stage-2 model) ===")
print(best_hurdle_per_h["Model"].value_counts().to_string())

print()
print("=== Does the hurdle structure help, per model? (Hurdle_NSE - SingleStage_NSE, averaged across H) ===")
hurdle_test_df["hurdle_minus_singlestage"] = hurdle_test_df["Hurdle_NSE"] - hurdle_test_df["SingleStage_NSE"]
print(hurdle_test_df.groupby("Model")["hurdle_minus_singlestage"].mean().to_string())

=== Hurdle_NSE by Model x Horizon ===
Model  CatBoost   LightGBM    XGBoost
H                                    
1      0.733898  -3.474663   0.661062
2      0.344548 -12.238057  -9.014738
3     -2.845024 -12.730680 -10.502554
4     -0.672978  -7.918782 -17.762052
5     -1.001499 -10.001693  -4.687095
6     -0.094929  -2.892402  -4.273086
7      0.253040  -1.764405  -3.016094

=== Stage 2 in_train_cv_nse by Model x Horizon (read this alongside Hurdle_NSE above --
    a model with poor in_train_cv_nse at a given horizon should not be trusted there,
    even if its test-set number happens to look acceptable) ===
Model  CatBoost  LightGBM   XGBoost
H                                  
1      0.750103  0.634207  0.769449
2      0.556158  0.200916  0.483740
3      0.307280 -0.052314  0.265622
4      0.095865  0.124579  0.047749
5     -0.676936 -0.811524  0.064630
6     -1.120273 -0.157335 -0.178817
7     -4.524015 -1.945145 -1.405587

=== Best Stage-2 model per horizon (by Hurdle_NSE) ===
 

### วิธีอ่านผลลัพธ์ Hurdle Model

**`in_train_cv_auc`** (ดูใน Stage 1 diagnostics) คือสัญญาณที่เชื่อถือได้ว่า Stage 1 แยกวัน zero-inflow ได้จริงหรือไม่ ภายใน training distribution -- ถ้าสูง (เคยพบ 0.88-0.96 ในรอบทดสอบก่อนหน้า) แสดงว่า feature มี signal จริง

**`val_auc` ไม่น่าเชื่อถือเท่า `in_train_cv_auc`** ถ้า val มี zero-rate ต่างจาก train มาก (ดูตัวเลขที่ Part 1)

**ผลบน TEST อาจเป็นแบบผสม ไม่ใช่ดีขึ้นทุก horizon** -- บาง horizon threshold อาจทำนาย "zero" เกินจริงหรือไม่พอ เพราะ train มีตัวอย่าง zero-inflow แค่ 6-10 แถวต่อ horizon ทำให้การเลือก threshold มี noise สูง **นี่เป็นข้อจำกัด จากปริมาณข้อมูล ไม่ใช่จาก algorithm หรือโมเดลที่เลือกใช้ใน Stage 2**

**เกี่ยวกับ Stage 2:** เดิม Stage 2 ใช้ early stopping กับ X_val เต็มชุด ซึ่งพบว่าพังจริง — Stage 2 เทรนเฉพาะแถว Q_in_actual>0 แต่ X_val มีวัน zero-inflow ปนอยู่ ทำให้ early stopping เก็บ iteration แรกสุดบ่อยๆ (โมเดลไม่เรียนรู้ อะไรเลย) **แก้แล้วด้วย fixed iterations จาก Optuna** พร้อมตรวจสอบด้วย in-train CV (`in_train_cv_nse`, CV บน train-positive เท่านั้น ไม่พึ่ง val) เป็นสัญญาณความน่าเชื่อถือ

**สิ่งที่ลองแล้วแต่ไม่ใช้ (สำคัญที่จะรู้):** เคยลองทำ auto-fallback ไปใช้ persistence อัตโนมัติเมื่อ `in_train_cv_nse` ต่ำเกินไป และเคยลองทำ "extrapolation ceiling" (จำกัดค่าทำนายไม่ให้เกินค่าที่เคยเห็นใน train) ทั้งสองวิธีถูกตัดออกหลังพิสูจน์ว่าใช้ไม่ได้จริง:
- `in_train_cv_nse` ไม่ใช่ตัวพยากรณ์ที่แม่นยำพอ — พบว่า XGBoost มี `in_train_cv_nse` อยู่ในช่วงที่ดูปกติ (-0.56 ถึง 0.75) ทุก horizon แต่ test NSE กลับพังถึง -20.7 ที่ H4 เพราะ train-positive มีแค่ ~220 แถว ไม่ครอบคลุม สภาวะที่เจอใน test
- "extrapolation ceiling" ใช้ไม่ได้เพราะข้อมูล train มี outlier ปีน้ำหลากรุนแรง (สูงสุดถึง 1,408,228 ทั้งที่ค่ากลาง ทั่วไปอยู่ที่ ~35,000-42,000) ทำให้ ceiling ที่อิงจาก percentile ของ train หลวมเกินจะจับอะไรได้เลย และที่สำคัญกว่า: คำทำนายที่ผิดจริงๆไม่ใช่ค่าที่ "เป็นไปไม่ได้" ทางกายภาพ — เป็นแค่ค่าที่ผิดในขนาดที่สมเหตุสมผล ไม่มี filter ตาม ช่วงค่าใดจะจับปัญหาความแม่นยำแบบนี้ได้ ต้องใช้ข้อมูลมากขึ้นหรือ feature/โมเดลที่ต่างออกไปเท่านั้น

**ข้อสรุป:** รายงาน `in_train_cv_nse` ไว้ตรงๆให้ตรวจสอบเอง ไม่ใช้ automatic threshold ใดๆ ตัดสินใจแทน — ดูคอลัมน์ `stage2_in_train_cv_nse` คู่กับ `Hurdle_NSE` เสมอ ถ้า horizon ไหนของโมเดลไหนมี `in_train_cv_nse` ต่ำ ให้ระมัดระวังตัวเลข test ของ horizon นั้นเป็นพิเศษ แม้ตัวเลข test จะดูดีก็ตาม

**ดู `hurdle_minus_singlestage`** เพื่อตอบคำถามว่า "การเพิ่ม Stage 1 ช่วยโมเดลนี้จริงไหม" แยกตามแต่ละโมเดล -- ถ้าค่าเป็นบวก แสดงว่าโครงสร้าง hurdle ช่วย ถ้าติดลบ แสดงว่า single-stage ของโมเดลนั้นยังดีกว่า

**คำแนะนำสำหรับ manuscript:** เสนอ hurdle เป็นแนวทางที่มีศักยภาพพร้อมหลักฐาน (`in_train_cv_auc`, `in_train_cv_nse`) ไม่ใช่ผลสำเร็จแล้ว -- ระบุ threshold instability และ Stage 2 fallback rate เป็น limitation ที่ผูกกับขนาดข้อมูล (มีข้อมูลมากกว่า 1 ปีน่าจะช่วยทั้งสองจุดได้)


In [15]:
# =============================================================================
# EXPORT MODELS FOR DEPLOYMENT (Reservoir Inflow — Hurdle Model)
# =============================================================================
# วางเซลล์นี้ต่อท้าย notebook แล้วรัน — ต้องรันหลังจากเซลล์ Stage 1, Stage 2,
# และเซลล์เปรียบเทียบ hurdle (best_hurdle_per_h) ทำงานเสร็จแล้ว เพราะโค้ดนี้
# ใช้ตัวแปรที่มีอยู่แล้วในหน่วยความจำ (stage1_models, stage1_thresholds,
# stage2_models, hurdle_test_df, feature_cols, targets ฯลฯ) ไม่ได้ retrain ใหม่
#
# ผลลัพธ์: โฟลเดอร์ exported_models/ พร้อมไฟล์ .pkl + metadata สำหรับให้
# pipeline (data_pipeline.py) โหลดไปใช้ทำนายจริงได้ทันที
# =============================================================================

import json
import platform
import sys
from datetime import datetime

import catboost
import joblib
import lightgbm
import sklearn
import xgboost
import os

EXPORT_DIR = "exported_models"
os.makedirs(EXPORT_DIR, exist_ok=True)

# -----------------------------------------------------------------------
# 1. Stage 1 — CatBoost classifiers (zero/non-zero) ต่อ horizon
#    รูปแบบ: dict {target_column_name: fitted CatBoostClassifier}
# -----------------------------------------------------------------------
joblib.dump(stage1_models, os.path.join(EXPORT_DIR, "stage1_classifiers.pkl"))
joblib.dump(stage1_thresholds, os.path.join(EXPORT_DIR, "stage1_thresholds.pkl"))

# -----------------------------------------------------------------------
# 2. Stage 2 — เก็บโมเดลทั้ง 3 ตัวไว้ครบ (สำหรับ audit/เทียบย้อนหลังได้)
#    รูปแบบ: dict {(model_name, target_column_name): fitted regressor}
# -----------------------------------------------------------------------
joblib.dump(stage2_models, os.path.join(EXPORT_DIR, "stage2_regressors_all_models.pkl"))

# -----------------------------------------------------------------------
# 3. เลือกโมเดลที่ดีที่สุดต่อ horizon ตาม Hurdle_NSE (ไม่ hardcode ชื่อโมเดล
#    เผื่ออนาคต retrain แล้วผลเปลี่ยน — อ่านจาก hurdle_test_df ที่คำนวณไว้แล้ว)
# -----------------------------------------------------------------------
best_per_h = (
    hurdle_test_df.loc[hurdle_test_df.groupby("H")["Hurdle_NSE"].idxmax(), ["H", "Model", "Hurdle_NSE"]]
    .set_index("H")
    .to_dict(orient="index")
)
# best_per_h ตัวอย่าง: {1: {"Model": "CatBoost", "Hurdle_NSE": 0.714483}, ...}

deployment_stage2 = {}          # {horizon(int): fitted regressor ที่ดีที่สุด}
deployment_model_choice = {}    # {horizon(int): ชื่อโมเดลที่เลือก}
for h, tcol in enumerate(targets, start=1):
    chosen_model_name = best_per_h[h]["Model"]
    deployment_stage2[h] = stage2_models[(chosen_model_name, tcol)]
    deployment_model_choice[h] = {
        "model_name": chosen_model_name,
        "hurdle_nse_on_test": best_per_h[h]["Hurdle_NSE"],
        "target_column": tcol,
    }

joblib.dump(deployment_stage2, os.path.join(EXPORT_DIR, "deployment_stage2_regressors.pkl"))

with open(os.path.join(EXPORT_DIR, "deployment_model_choice.json"), "w", encoding="utf-8") as f:
    json.dump(deployment_model_choice, f, ensure_ascii=False, indent=2)

# -----------------------------------------------------------------------
# 4. Metadata — feature schema + logic ที่ pipeline ต้องใช้ตอนสร้าง feature
#    (ตรงนี้คือส่วนที่ก่อนหน้านี้ "ไม่มีไฟล์แยก" — export ให้ตอนนี้เลย)
# -----------------------------------------------------------------------
metadata = {
    "exported_at": datetime.now().isoformat(),
    "target_reservoir": "ระบุชื่ออ่างเก็บน้ำที่ train ด้วยตรงนี้",  # TODO: ใส่ชื่ออ่างจริง
    "feature_cols": feature_cols,
    "targets": targets,
    "date_col_raw": "Date",
    "qin_col": QIN_COL,
    "rain_col": RAIN_COL,
    "api_col": API_COL,
    "zero_threshold": ZERO_THRESHOLD,
    "horizons": list(range(1, len(targets) + 1)),
    "model_structure": {
        "stage1": "CatBoostClassifier ทำนาย P(Q_in_t+h = 0), threshold ต่อ horizon ใน stage1_thresholds.pkl",
        "stage2": "Regressor ทำนาย DELTA = Q_in_t+h(actual) - Q_in_t(ปัจจุบัน) ไม่ใช่ค่าจริงตรงๆ",
        "final_prediction_logic": (
            "ถ้า stage1_classifier.predict_proba(X)[:,1] >= stage1_thresholds[h] "
            "=> prediction = 0 "
            "มิฉะนั้น => prediction = clip(Q_in_t(ปัจจุบัน) + stage2_regressor.predict(X), 0, None)"
        ),
        "stage2_trained_on": "เฉพาะแถวที่ Q_in_actual > ZERO_THRESHOLD ใน train เท่านั้น",
    },
    "deployment_model_per_horizon": deployment_model_choice,
    "training_data_source": "Training_Values_Nofct_7day_CORRECTED.csv",
    "training_date_range": {
        "start": str(dates.min()),
        "end": str(dates.max()),
    },
    "split_method": "stratified season-balanced split (70/15/15), ไม่ใช่ chronological split ตรงๆ",
    "known_limitations": [
        "Test set นี้ไม่สามารถเทียบ RMSE ข้าม notebook อื่นได้โดยตรง (scale/composition ต่างกัน)",
        "Stage 1 มีตัวอย่าง zero-inflow ใน train แค่ 6-10 แถวต่อ horizon ทำให้ threshold มี noise สูง",
        "Horizon 3-7 มี Hurdle_NSE ติดลบสำหรับบางโมเดล ควรระวังเป็นพิเศษเมื่อนำไปใช้จริง "
        "(ดู deployment_model_choice.json ค่า hurdle_nse_on_test ประกอบก่อนเชื่อผลทำนาย)",
    ],
    "library_versions": {
        "python": sys.version,
        "platform": platform.platform(),
        "catboost": catboost.__version__,
        "xgboost": xgboost.__version__,
        "lightgbm": lightgbm.__version__,
        "scikit_learn": sklearn.__version__,
    },
}

with open(os.path.join(EXPORT_DIR, "model_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)

# -----------------------------------------------------------------------
# 5. Performance summary (เก็บผลประเมินทั้งหมดไว้คู่กับโมเดล เพื่อ traceability)
# -----------------------------------------------------------------------
hurdle_test_df.to_csv(os.path.join(EXPORT_DIR, "model_performance_summary.csv"), index=False)

# -----------------------------------------------------------------------
# 6. Sanity check — โหลดกลับมาทันทีแล้วทดสอบทำนาย 1 รอบ เพื่อยืนยันว่าไฟล์ที่
#    export ใช้งานได้จริง ไม่ใช่แค่ save ผ่านเฉยๆ
# -----------------------------------------------------------------------
print("=" * 70)
print("VERIFYING EXPORTED FILES...")
print("=" * 70)

_stage1 = joblib.load(os.path.join(EXPORT_DIR, "stage1_classifiers.pkl"))
_thresholds = joblib.load(os.path.join(EXPORT_DIR, "stage1_thresholds.pkl"))
_stage2_deploy = joblib.load(os.path.join(EXPORT_DIR, "deployment_stage2_regressors.pkl"))

# ทดสอบทำนาย horizon 1 ด้วยแถวแรกของ X_test เทียบกับค่าที่คำนวณไว้แล้วในเซลล์เดิม
h_test, tcol_test = 1, targets[0]
X_sample = X_test.iloc[[0]]
qin_now = X_sample[QIN_COL].to_numpy()[0]

clf_loaded = _stage1[tcol_test]
thr_loaded = _thresholds[tcol_test]
reg_loaded = _stage2_deploy[h_test]

is_zero_pred = clf_loaded.predict_proba(X_sample)[:, 1][0] >= thr_loaded
if is_zero_pred:
    y_pred_reloaded = 0.0
else:
    delta_pred = reg_loaded.predict(X_sample)[0]
    y_pred_reloaded = max(qin_now + delta_pred, 0.0)

print(f"H{h_test} — ทำนายด้วยโมเดลที่โหลดจากไฟล์ .pkl: {y_pred_reloaded:,.1f} m3/day")
print(f"  (เทียบกับค่าจริง ณ แถวนี้: {Y_test_abs[tcol_test].iloc[0]:,.1f} m3/day)")
print(f"  Model ที่ใช้ H1: {deployment_model_choice[1]['model_name']} "
      f"(test Hurdle_NSE = {deployment_model_choice[1]['hurdle_nse_on_test']:.3f})")

print()
print("=" * 70)
print(f"EXPORT COMPLETE -> {EXPORT_DIR}/")
print("=" * 70)
for fname in sorted(os.listdir(EXPORT_DIR)):
    fpath = os.path.join(EXPORT_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname:45s} {size_kb:>10.1f} KB")


VERIFYING EXPORTED FILES...
H1 — ทำนายด้วยโมเดลที่โหลดจากไฟล์ .pkl: 66,394.7 m3/day
  (เทียบกับค่าจริง ณ แถวนี้: 69,911.7 m3/day)
  Model ที่ใช้ H1: CatBoost (test Hurdle_NSE = 0.734)

EXPORT COMPLETE -> exported_models/
  deployment_model_choice.json                         1.0 KB
  deployment_stage2_regressors.pkl                  1704.9 KB
  model_metadata.json                                  3.8 KB
  model_performance_summary.csv                        5.5 KB
  stage1_classifiers.pkl                             318.3 KB
  stage1_thresholds.pkl                                0.2 KB
  stage2_regressors_all_models.pkl                  9791.3 KB
